In [ ]:
VERSION = "1.0.1_sextractor"

# "1.0.1": "ellips gaussian + pixels_out",
# "1.0.2": "ellips uniform + pixels_out",
# "1.0.3": "ellips gaussian",
# hypercube + 40_000 + shear round 2 + 10 gal/patch + shear |0.06|

# lib

In [ ]:
from pathlib import Path

from jax import numpy
from polars import DataFrame, read_csv
from tqdm import tqdm
from utils import (
    check_psf,
    compute_ellipticities,
    compute_ellipticities_quadratic,
    final_results,
    histos,
    load_and_slice_csv,
    plot_sersic,
    plot_vignettes,
    print_fits,
)

DATASET_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / VERSION
VERSION_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / VERSION
CATALOG_DIR = VERSION_DIR / "CATALOG"
IMAGES_DIR = VERSION_DIR / "IMAGES"
PLOTS_DIR = VERSION_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# sersic (numerical stability?)

In [ ]:
X_min, X_max = -2.5, 1.5
plot_sersic(X_min, X_max, PLOTS_DIR)
del X_min, X_max

# psf

In [ ]:
check_psf(
    Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / "PSF" / "airy.psf.fits", PLOTS_DIR
)

# images

In [ ]:
CHOICE = "simu_00005"

print_fits(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"background_{CHOICE}",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
    ],
    ["image", "fitting", "background", "residual", "segmentation"],
    PLOTS_DIR,
)

# vignettes

In [ ]:
CHOICE = "simu_00001"

plot_vignettes(
    PLOTS_DIR,
    csv_file=CATALOG_DIR / f"catalog_{CHOICE}.csv",
    vignettes_file=IMAGES_DIR / f"vignettes_{CHOICE}.fits",
)

# inference

In [ ]:
RESULTS_FILES = sorted(list((CATALOG_DIR).glob("catalog_simu_*.csv")))
NB_TOTAL = len(RESULTS_FILES)

detections, avg_e1, avg_e2, avg_e1_quad, avg_e2_quad = [], [], [], [], []
for sample in tqdm(RESULTS_FILES, total=NB_TOTAL):
    df = read_csv(sample)
    detections.append(df.shape[0])

    a = df["ellipse_a"].to_numpy()
    b = df["ellipse_b"].to_numpy()
    theta = df["ellipse_theta"].to_numpy()

    cxx = df["ellipse_cxx"].to_numpy()
    cyy = df["ellipse_cyy"].to_numpy()
    cxy = df["ellipse_cxy"].to_numpy()

    e1, e2 = compute_ellipticities(a, b, theta)
    avg_e1.append(e1.mean())
    avg_e2.append(e2.mean())

    e1_quad, e2_quad = compute_ellipticities_quadratic(cxx, cyy, cxy)
    avg_e1_quad.append(e1_quad.mean())
    avg_e2_quad.append(e2_quad.mean())


file_path = VERSION_DIR / "pred_avg_ellipticies.csv"
DataFrame(
    {
        "avg_e1": avg_e1,
        "avg_e2": avg_e2,
        "avg_e1_quad": avg_e1_quad,
        "avg_e2_quad": avg_e2_quad,
    },
    schema=["avg_e1", "avg_e2", "avg_e1_quad", "avg_e2_quad"],
).write_csv(file_path)

del (
    df,
    avg_e1,
    avg_e2,
    avg_e1_quad,
    avg_e2_quad,
    a,
    b,
    theta,
    cxx,
    cxy,
    cyy,
    e1,
    e2,
    e1_quad,
    e2_quad,
)

# postprocess

In [ ]:
pred_avg_e = load_and_slice_csv(VERSION_DIR / "pred_avg_ellipticies.csv", NB_TOTAL)
true_avg_e = load_and_slice_csv(DATASET_DIR / "true_avg_ellipticies.csv", NB_TOTAL)
true_shears = load_and_slice_csv(DATASET_DIR / "true_avg_shears.csv", NB_TOTAL)
print(pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

for i in range(2):
    pred_avg_e = pred_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_avg_e = true_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_shears = true_shears[numpy.abs(true_shears[:, i]) <= 0.05]
print("restricted range", pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

# histo

In [ ]:
histos(
    diff_e1=pred_avg_e[:, 0] - pred_avg_e[:, 2],
    diff_e2=pred_avg_e[:, 1] - pred_avg_e[:, 3],
    data_gal=detections,
    directory=PLOTS_DIR,
    bins=10,
)

# results

In [ ]:
final_results(true_avg_e, pred_avg_e[:, :2], true_shears, PLOTS_DIR / "results.pdf")

# results recons

In [ ]:
final_results(true_avg_e, pred_avg_e[:, -2:], true_shears, PLOTS_DIR / "results_quad.pdf")